In [ ]:
import socket
import cv2
import numpy as np
import time

from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")
btns = base.btns_gpio
leds = base.leds

### Functions

In [ ]:
def bgr_to_lab(color):
    bgr = np.array(color, dtype=np.uint8).reshape(1,1,3)  # shape (1,1,3)
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    return tuple(lab[0,0])

In [ ]:
def bgr_to_hsv(color):
    bgr = np.array(color, dtype=np.uint8).reshape(1,1,3)  # shape (1,1,3)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    return tuple(hsv[0,0])

In [ ]:
def determine_season_lab(colors):

    skin, hair, eye = colors

    sL,sA,sB = skin
    hL,hA,hB = hair
    eL,eA,eB = eye

    # ---- temperature ----
    warm_score = (sA-128) + (sB-128)
    hue = "warm" if warm_score > 0 else "cool"

    # ---- chroma ----
    def chroma(a,b):
        return np.sqrt((a-128)**2 + (b-128)**2)

    avg_chroma = (
        chroma(sA,sB) +
        chroma(hA,hB) +
        chroma(eA,eB)
    ) / 3

    chroma_type = "bright" if avg_chroma > 25 else "muted"

    # ---- contrast ----
    contrast = max(sL,hL,eL) - min(sL,hL,eL)

    contrast_type = "high" if contrast > 80 else "low"

    # ---- season ----
    if hue=="warm" and chroma_type=="bright":
        return "Spring"

    if hue=="warm" and chroma_type=="muted":
        return "Autumn"

    if hue=="cool" and chroma_type=="muted":
        return "Summer"

    if hue=="cool" and chroma_type=="bright":
        return "Winter"

In [ ]:
def classify_season(hsv_colors):

    skin_h, skin_s, skin_v = hsv_colors[0]
    hair_h, hair_s, hair_v = hsv_colors[1]
    eye_h, eye_s, eye_v = hsv_colors[2]

    # --- HUE (warm/cool) ---
    warm_score = skin_h
    print(f"hue for skin {skin_h}, hair {hair_h}, eyes {eye_h}")
    print(f"warmth score: {warm_score}")
    hue = "warm" if warm_score >= 2 else "cool"

    # --- CHROMA (bright/muted) ---
    avg_sat = (skin_s + hair_s + eye_s) / 3
    print(f"saturation for skin {skin_s}, hair {hair_s}, eyes {eye_s}")
    print(f"saturation avg: {avg_sat}")
    chroma = "bright" if avg_sat > 110 else "muted"

    # --- VALUE / CONTRAST ---
    contrast = max(skin_v, hair_v, eye_v) - min(skin_v, hair_v, eye_v)
    value = "high" if contrast > 90 else "low"
    print(f"brightness for skin {skin_v}, hair {hair_v}, eyes {eye_v}")
    print(f"contrast: {value}")

    # --- SEASON ---
    if hue == "warm" and chroma == "bright":
        return "spring"
    if hue == "warm" and chroma == "muted":
        return "autumn"
    if hue == "cool" and chroma == "muted":
        return "summer"
    if hue == "cool" and chroma == "bright":
        return "winter"

    return season

In [ ]:
 def all_onboard_leds_off():
    for a in leds:
        a.off()

### Main

In [ ]:
# skin, hair, eye in BGR format
all_onboard_leds_off()

CLIENT_IP = "0.0.0.0"
LISTENING_PORT = 9999

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

sock.bind((CLIENT_IP, LISTENING_PORT))

sock.listen(1)

conn, addr = sock.accept()
print(f"Accepted connection from {addr}")

colors = []
while True:
    data = conn.recv(1024)
    bgr_tuple = tuple(map(int, data.split()))
    colors.append(bgr_tuple)
    if not data:
        print("Client disconneted")
        break
    print("Received message: ", data.decode('utf8'))

conn.close()


# skin, hair, eye in HSV format
hsv_colors = []

for color in colors:
    hsv_colors.append(bgr_to_hsv(color))
    
season = classify_season(hsv_colors)
print(season)

led_num = -1

if season == 'spring':
    led_num = 0
elif season == 'summer':
    led_num = 1
elif season == 'autumn':
    led_num = 2
elif season == 'winter':
    led_num = 3
else: pass

for i in range(4):
    leds[led_num].on()
    time.sleep(0.5)
    leds[led_num].off()
    time.sleep(0.5)

leds[led_num].on()

print("Done.")
 